# Day 22: System Design Fundamentals

**Week 4 — System Design**

---

## 📚 Theory: The Google System Design Framework

A System Design interview is fundamentally different from a coding interview. There is no "compilable" right answer. It is a collaborative discussion where you design a scalable architecture to solve an open-ended problem.

Always follow this **5-Step Framework**:
1. **Requirements Clarification (5 mins)**: Determine Functional (what the system does) and Non-Functional (scale, latency, availability) requirements.
2. **API Design (5 mins)**: Define the core endpoints (e.g., `POST /create`, `GET /read`).
3. **High-Level Architecture (10 mins)**: Draw the "boxes and arrows". (Client -> Load Balancer -> Web Servers -> Database).
4. **Detailed Design / Deep Dive (15 mins)**: Zoom into the hardest parts. (How do we shard the database? How do we handle cache invalidation?)
5. **Bottlenecks & Trade-offs (5 mins)**: Discuss Single Points of Failure (SPOF) and alternative technologies.

### Core Infrastructure Components
- **Load Balancers (LB)**: Distributes incoming network traffic across multiple servers to ensure no single server bears too much demand.
- **API Gateway**: A server that acts as an API front-end, receiving API requests, enforcing throttling, and routing them to the internal microservices.
- **Caches (Redis/Memcached)**: Extremely fast in-memory databases used to store frequently accessed data.
- **CDN (Content Delivery Network)**: Globally distributed proxy servers that cache static assets (images, videos, HTML) closer to users.
- **Message Queues (Kafka/RabbitMQ)**: Decouples heavy processing tasks by placing them in an asynchronous queue.

### The CAP Theorem
In a distributed system, you can only pick 2 of the following 3:
- **Consistency**: Every read receives the most recent write or an error.
- **Availability**: Every request receives a (non-error) response, without the guarantee that it contains the most recent write.
- **Partition Tolerance**: The system continues to operate despite an arbitrary number of messages being dropped or delayed by the network.

*(Note: Because network partitions (P) are unavoidable in real life, you usually have to choose between **CP** (Consistency) and **AP** (Availability).)*

## 🔨 Exercise 1: Design a URL Shortener (TinyURL)

This is the "Hello World" of System Design. Treat this markdown cell as your whiteboard.

### 1. Requirements
**Functional**:
1. Given a URL, our service should generate a shorter and unique alias of it.
2. When users access a short link, our service should redirect them to the original link.

**Non-Functional**:
1. Highly Available (If it goes down, millions of links break).
2. URL redirection should happen in real-time with minimal latency.
3. Short links should not be predictable.

**Traffic Estimates**:
- Let's assume 100 million new URLs generated per month.
- 100M / (30 * 24 * 3600) = ~40 URLs/second writes.
- Assume 100:1 read/write ratio = 4,000 URLs/second reads.

### 2. API Design
Design the REST API endpoints below:

In [ ]:
# Write your proposed API endpoints in python pseudo-code/docstrings

def create_short_url(api_dev_key, original_url, custom_alias=None):
    """
    YOUR API DESIGN HERE
    """
    pass

def get_url(api_dev_key, short_url):
    """
    YOUR API DESIGN HERE
    """
    pass


### 3. Database Schema

Since we need to store billions of rows, but the rows are extremely small (just `hash` -> `long_url`) and have virtually no complex relational joins, a **NoSQL Key-Value Store** (like DynamoDB or Cassandra) is highly preferred for scale, though a sharded SQL database could also work.

**Table: URL_Map**
- `hash` (varchar, Primary Key) - The short string (e.g. "aB3x9")
- `original_url` (varchar)
- `creation_date` (datetime)
- `expiration_date` (datetime)
- `user_id` (int, indexed)

### 4. High-Level Architecture

```mermaid
graph TD
    Client(Client) --> LB[Load Balancer]
    LB --> Web1[Web Server]
    LB --> Web2[Web Server]
    Web1 --> Cache[(Redis Cache)]
    Web2 --> Cache
    Web1 --> DB[(NoSQL Database)]
    Web2 --> DB
```

**Read Flow (Redirection)**:
1. User clicks `tinyurl.com/aB3x9`.
2. Request hits Load Balancer, routes to Web Server.
3. Web Server checks **Redis Cache** for `aB3x9`.
4. If Cache Miss, Web Server queries Database, updates Cache, and returns HTTP 301 (Permanent Redirect) or HTTP 302 (Temporary Redirect) to the original URL.

**Write Flow (Creation)**:
1. How do we generate the short string? 
2. **Approach A (Base62 Encoding)**: Hash the original URL using MD5, take the first 43 bits, and convert to Base62 (A-Z, a-z, 0-9). Issue: Collisions! Multiple URLs could hash to the same value.
3. **Approach B (Key Generation Service - KGS)**: Pre-generate random 6-character strings offline and store them in a database. When a new URL comes in, just pop a pre-generated string from the KGS. This guarantees no collisions and makes writes `O(1)` fast!

## 📝 Day 22 Review

### Concepts Validated Today
- [ ] The 5-Step System Design Interview framework.
- [ ] CAP Theorem and understanding when to prioritize Availability over Consistency.
- [ ] The difference between an API Gateway and a Load Balancer.
- [ ] How to design an `O(1)` read/write architecture for URL shortening using Caches and a Key Generation Service.

### Tomorrow's Preview
**Day 23: Designing Distributed Systems** — We will tackle heavier consistency problems like Rate Limiters and Distributed Key-Value stores.